# Malus's Law Verification

Optics TP — L3 Physics, Sorbonne Université

Checking Malus's law with a polarized He-Ne laser (λ = 633 nm) and a rotating analyzer.  
The photodiode gives a voltage proportional to transmitted intensity.

Recall: $I(\theta) = I_0 \cos^2(\theta)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from optics import polarization, visualization

print(f"Toolkit version: {sys.modules['optics'].__version__}")

## Experimental data

Photodiode voltage (mV) vs analyzer angle.

In [ ]:
# analyzer angle (degrees) and measured voltage (mV)
angle_deg = np.array([92, 88, 84, 80, 76, 70, 60, 50, 40, 30, 20, 10, 0])
tension = np.array([0.0, 1.7, 5.4, 8.2, 12.8, 16.8, 29.2, 44.5, 60.7, 75.7, 85.7, 91.3, 91.7])

# uncertainties
erreurs_angle_deg = 1.0  # +/-1 deg
erreurs_tension = 0.1    # +/-0.1 mV

print(f"{len(angle_deg)} points, angle from {angle_deg.min()} to {angle_deg.max()} deg")
print(f"Voltage from {tension.min():.1f} to {tension.max():.1f} mV")

## Quick look at the data

In [ ]:
plt.figure(figsize=(9, 5))
plt.errorbar(angle_deg, tension,
             xerr=erreurs_angle_deg, yerr=erreurs_tension,
             fmt='o', capsize=4, label='data')
plt.xlabel('Analyzer angle (deg)')
plt.ylabel('Voltage (mV)')
plt.title('Intensity vs angle')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# looks like cos^2, let's fit

## Fit

Fitting $I(\theta) = I_0 \cos^2(\theta) + \text{offset}$ — the offset accounts for background light / dark current.

In [ ]:
res = polarization.fit_malus_law(
    angle_deg,
    tension,
    angle_uncertainties=np.full_like(angle_deg, erreurs_angle_deg),
    intensity_uncertainties=np.full_like(tension, erreurs_tension)
)

print(f"I0 = {res['I0']:.2f} +/- {res['u_I0']:.2f} mV")
print(f"offset = {res['offset']:.3f} +/- {res['u_offset']:.3f} mV")
print(f"R2 = {res['R2']:.6f}")

R2 > 0.99, good fit.

## Data + fit + residuals

In [ ]:
# plot data and fitted curve
theta_fine = np.linspace(0, 92, 200)
theta_fine_rad = np.deg2rad(theta_fine)
I_fit = polarization.predict_malus(res, theta_fine_rad)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7),
                               gridspec_kw={'height_ratios': [3, 1]},
                               sharex=True)

ax1.errorbar(angle_deg, tension,
             xerr=erreurs_angle_deg, yerr=erreurs_tension,
             fmt='o', capsize=4, label='data')
ax1.plot(theta_fine, I_fit, 'r-', lw=2, label='fit')
ax1.set_ylabel('Voltage (mV)')
ax1.set_title("Malus's law -- He-Ne laser")
ax1.legend()
ax1.grid(alpha=0.3)

# residuals
predicted = polarization.predict_malus(res, np.deg2rad(angle_deg))
residuals = tension - predicted

ax2.plot(angle_deg, residuals, 'o')
ax2.axhline(0, color='red', ls='--', lw=1)
ax2.set_xlabel('Angle (deg)')
ax2.set_ylabel('Residual (mV)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Residuals

In [ ]:
print(f"Mean bias:    {np.mean(residuals):.4f} mV")
print(f"Std dev:      {np.std(residuals):.4f} mV")
print(f"Max |resid|:  {np.max(np.abs(residuals)):.4f} mV")

# fairly random, no obvious trend
# scatter is larger than 0.1 mV though
# probably stray light, vibrations, etc.

## Degree of polarization

In [ ]:
I_max = res['I0'] + res['offset']   # theta = 0
I_min = res['offset']                # theta = 90

P, u_P = polarization.calculate_polarization_degree(
    I_max, I_min,
    u_I_max=res['u_I0'],
    u_I_min=res['u_offset']
)

print(f"I_max = {I_max:.2f} mV, I_min = {I_min:.2f} mV")
print(f"Polarization degree: P = {P:.4f} +/- {u_P:.4f} ({P*100:.1f}%)")
print(f"Extinction ratio: {I_max/I_min:.0f}:1")

P > 99%, laser is very well polarized — expected for He-Ne.

## Comparison with theory

In [ ]:
# quick check: at 45 deg we expect ~50% of I0
pred_45 = polarization.predict_malus(res, np.deg2rad(np.array([45])))[0]
print(f"Prediction at 45 deg: {pred_45:.2f} mV ({100*pred_45/res['I0']:.1f}% of I0)")

In [ ]:
theta_fine = np.linspace(0, 90, 200)

I_ideal = res['I0'] * np.cos(np.deg2rad(theta_fine))**2  # no offset
I_fit = polarization.predict_malus(res, np.deg2rad(theta_fine))  # with offset

plt.figure(figsize=(9, 5))
plt.errorbar(angle_deg, tension, yerr=erreurs_tension,
             fmt='o', capsize=4, label='data', alpha=0.8)
plt.plot(theta_fine, I_ideal, 'g--', lw=2, label='ideal cos^2 (no offset)', alpha=0.6)
plt.plot(theta_fine, I_fit, 'r-', lw=2, label='fit (with offset)')
plt.xlabel('Analyzer angle (deg)')
plt.ylabel('Voltage (mV)')
plt.title('Data vs theory')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Offset is {res['offset']:.3f} mV = {100*res['offset']/res['I0']:.2f}% of I0, basically negligible.")

## Conclusion

Data follows Malus's law well (R2 > 0.99). The offset is tiny and the degree of polarization is >99%, consistent with a He-Ne laser.

Residuals don't show a clear systematic trend, but they're a bit larger than the instrumental uncertainty — probably stray light, slight misalignment, that sort of thing.

Overall the polarizer-analyzer setup works as expected.

---
*Optical Analysis Toolkit — data from L3 Physics optics TP, Sorbonne Université*